# LangGraph — Analyst → Strategist pipeline

LangGraph is LangChain's graph-based orchestration library: you model a workflow as a
**directed graph**, where nodes are functions (often LLM calls) and edges are the
transitions between them. It supports branching, loops, and persistent state — you're
writing the wiring diagram yourself.

| Framework | Core mental model | How multi-step flow is defined | Orchestration style |
|---|---|---|---|
| **LangGraph** | Explicit state graph | You wire nodes (functions/LLM calls) and edges yourself | Manual, graph-based — full control, more code |
| CrewAI | Role-based crew | You define `Agent`s (role/goal) and `Task`s; a `Process` (sequential/hierarchical) runs them | Opinionated, role-based — less code, less control |
| Microsoft Agent Framework | Unified successor to AutoGen + Semantic Kernel | Agents come from a chat client; you wire them with a Builder (Sequential, Concurrent, Handoff, GroupChat, Magentic) | Both — low-level like LangGraph, or pre-built patterns like CrewAI |



In [ ]:
%pip install -q langgraph langchain-anthropic anthropic

> **Gotcha:** After the install cell finishes for the first time, use
> **Runtime ▸ Restart session** before running the rest of the notebook. Colab can keep
> an old, incompatible copy of a dependency loaded in memory even after a fresh install
> replaces it on disk.


In [ ]:
import os

def _load(key, default=None):
    """Colab Secrets first, then environment variable, then interactive prompt."""
    try:
        from google.colab import userdata
        val = userdata.get(key)
        if val:
            return val
    except Exception:
        pass
    val = os.environ.get(key, default)
    if not val:
        from getpass import getpass
        val = getpass(f"{key}: ")
    return val

ANTHROPIC_API_KEY = _load("ANTHROPIC_API_KEY")
ANTHROPIC_MODEL = _load("ANTHROPIC_MODEL", "claude-sonnet-4-5-20250929")
os.environ["ANTHROPIC_API_KEY"] = ANTHROPIC_API_KEY

print(f"Using model: {ANTHROPIC_MODEL}")


## The scenario

An assistant reads user interviews, support tickets, and
competitor notes, and is supposed to turn that into a feature recommendation — without
mismatched analysis or hallucinated suggestions.

We'll build that as a **two-step pipeline**:

1. **Analyst** — reads the raw signals, extracts 2–3 themes, and cites which signal(s)
   support each one (a small guardrail against fabrication).
2. **Strategist** — takes *only* the Analyst's themes and recommends ONE feature with a
   one-line rationale. It's told not to invent anything the Analyst didn't surface.

Same signals, same two-step shape, three different frameworks — that's the point of
building all three tonight.

In LangGraph terms: two **nodes** (`analyst`, `strategist`), wired by two **edges**, sharing one **state** dict.


In [ ]:
SIGNALS = """
- User interview (Maria, SMB owner): "I got lost during setup and gave up before finishing."
- Support ticket #4821: "Can't find where to invite my team members. Confusing menu."
- Competitor teardown (RivalApp): just shipped a guided setup checklist with a progress bar.
- Support ticket #4903: "Wish there was a quick tour when I first logged in."
""".strip()

print(SIGNALS)


In [ ]:
from typing import TypedDict
from langchain_anthropic import ChatAnthropic

llm = ChatAnthropic(model=ANTHROPIC_MODEL, max_tokens=400)

class PipelineState(TypedDict):
    signals: str
    themes: str
    recommendation: str

def analyst_node(state: PipelineState) -> dict:
    prompt = (
        "You are a product signal analyst. Extract 2-3 themes from these signals "
        "as bullet points. Cite which signal supports each theme.\n\n"
        f"Signals:\n{state['signals']}"
    )
    response = llm.invoke(prompt)
    return {"themes": response.content}

def strategist_node(state: PipelineState) -> dict:
    prompt = (
        "You are a product strategist. Using ONLY the themes below, recommend ONE "
        "feature to prioritize next sprint with a one-line rationale. Do not invent "
        "new signals.\n\n"
        f"Themes:\n{state['themes']}"
    )
    response = llm.invoke(prompt)
    return {"recommendation": response.content}


In [ ]:
from langgraph.graph import StateGraph, START, END

graph = StateGraph(PipelineState)
graph.add_node("analyst", analyst_node)
graph.add_node("strategist", strategist_node)
graph.add_edge(START, "analyst")
graph.add_edge("analyst", "strategist")
graph.add_edge("strategist", END)

app = graph.compile()

result = app.invoke({"signals": SIGNALS, "themes": "", "recommendation": ""})
print("THEMES:\n", result["themes"])
print("\nRECOMMENDATION:\n", result["recommendation"])


## How LangGraph differs from CrewAI and Microsoft Agent Framework

| Framework | Core mental model | How multi-step flow is defined | Orchestration style |
|---|---|---|---|
| **LangGraph** | Explicit state graph | You wire nodes (functions/LLM calls) and edges yourself | Manual, graph-based — full control, more code |
| CrewAI | Role-based crew | You define `Agent`s (role/goal) and `Task`s; a `Process` (sequential/hierarchical) runs them | Opinionated, role-based — less code, less control |
| Microsoft Agent Framework | Unified successor to AutoGen + Semantic Kernel | Agents come from a chat client; you wire them with a Builder (Sequential, Concurrent, Handoff, GroupChat, Magentic) | Both — low-level like LangGraph, or pre-built patterns like CrewAI |


**Networking analogy:** LangGraph is like hand-building a network topology with static
routes. You declare every router (node) and every link (edge) yourself, and traffic
(state) only ever travels the path you cabled — including a deliberate loop back to an
earlier node if you want iterative refinement. Nothing moves that you didn't wire.

CrewAI trades that manual wiring for job descriptions and a process (see that notebook).
Microsoft Agent Framework gives you *both* — the same low-level node-and-edge control
(its `WorkflowBuilder`) plus pre-built patterns like the `SequentialBuilder` we used to
skip the manual graph entirely.

**Try it yourself:** add a third node — a `guardrail_node` that checks the
recommendation doesn't mention a signal that isn't in `SIGNALS`, and loop back to
`strategist` if it does.
